# 第四阶段：STL（标准模板库）

## 实验 4：`std::array` —— 固定长度的拥有型容器

实验 3 中的 `std::vector<T>` 拥有可变长度的连续存储；`std::array<T, N>` 同样拥有连续元素，但元素数量 `N` 是类型的一部分，在编译期已经确定。

本实验关注四个问题：

- 固定长度如何影响初始化、复制和接口设计；
- 为什么“内联存储”不等于“永远在栈上”；
- pointer/reference/iterator 的生命周期与 vector 有何不同；
- 固定长度数据如何映射到 C ABI 和 Kotlin/Native。

In [ ]:
// 本步骤：引入本实验需要的标准库和公开头文件。
#include <algorithm>
#include <array>
#include <cassert>
#include <cstddef>
#include <cstdint>
#include <iostream>
#include <numeric>
#include <stdexcept>
#include <tuple>
#include <vector>

### 1. 长度是类型的一部分

`std::array<int, 4>` 拥有恰好四个 `int` 元素。它通常可以理解为把原生数组包装成标准容器接口：

```text
std::array<T, N> owner
        │ contains N elements inline
        ▼
┌─────┬─────┬─────┬─────┐
│ T 0 │ T 1 │ ... │ T N │
└─────┴─────┴─────┴─────┘
size == N, no capacity, no reallocation
```

“内联”表示元素存储属于 array 对象自身，而不是像 vector 那样另行管理动态存储。array 作为局部变量时通常位于自动存储区；作为成员、静态对象或动态对象的一部分时，则跟随其 owner 所在位置，因此不要简单记成“array 一定在栈上”。

In [ ]:
// 本步骤：通过代码演示“长度是类型的一部分”并观察结果。
{
    // 创建长度写入类型的 array，并用四个值完整初始化。
    std::array<int, 4> values{10, 20, 30, 40};

    // 在编译期从类型中读取长度，验证 N 是类型信息的一部分。
    static_assert(
        std::tuple_size_v<decltype(values)> == 4);

    // 运行时输出固定大小和首地址，观察标准容器接口。
    std::cout << "size = " << values.size() << '\n';
    std::cout << "data = "
              << static_cast<const void *>(values.data())
              << '\n';

    assert(values.size() == 4);
}

### 2. 使用花括号明确初始化状态

`std::array` 是聚合类型。空花括号会值初始化全部元素；只给出部分初始值时，剩余元素也会值初始化。

不要读取未初始化的基础类型元素。下面的写法虽然能声明对象，但其中的 `int` 值不确定，因此只作为反例阅读：

```cpp
std::array<int, 4> uninitialized;
std::cout << uninitialized[0]; // undefined behavior
```

In [ ]:
// 本步骤：通过代码演示“使用花括号明确初始化状态”并观察结果。
{
    // 分别演示全部值初始化和部分初始化后的零填充。
    std::array<int, 4> zeros{};
    std::array<int, 4> partial{10, 20};

    // 验证基础类型元素都处于确定状态，而不是未初始化值。
    assert((zeros == std::array<int, 4>{0, 0, 0, 0}));
    assert((partial == std::array<int, 4>{10, 20, 0, 0}));

    // 输出部分初始化的结果，观察剩余元素被初始化为 0。
    std::cout << "partial =";

    for (int value : partial)
    {
        std::cout << ' ' << value;
    }

    std::cout << '\n';
}

### 3. 元素连续，但借用仍受 owner 生命周期约束

和 vector 一样，array 的元素连续排列，`data() + index` 与 `&array[index]` 指向同一元素。array 没有扩容操作，因此普通元素修改不会造成重分配；只要 owner 仍然存活，已有元素地址通常保持不变。

不过 `data()`、reference 和 iterator 仍然只是借用。array 析构后，它们同样会悬空。固定长度解决的是结构变化问题，不会自动延长 owner 生命周期。

In [ ]:
// 本步骤：通过代码演示“元素连续，但借用仍受 owner 生命周期约束”并观察结果。
{
    // 创建 owner，并在修改前借用首元素地址。
    std::array<int, 4> values{10, 20, 30, 40};
    int *first = values.data();

    // 逐元素比较 data + index 与下标地址，验证连续布局。
    for (std::size_t index = 0; index < values.size(); ++index)
    {
        assert(values.data() + index == &values[index]);

        std::cout << "values[" << index << "] = "
                  << values[index]
                  << ", address = "
                  << static_cast<const void *>(&values[index])
                  << '\n';
    }

    // 只修改元素值，不改变容器结构或元素地址。
    values.fill(7);

    // 验证修改后旧借用仍有效，并能看到更新后的首元素。
    assert(first == values.data());
    assert(*first == 7);
}

### 4. 固定长度不代表只读

array 不能 `push_back()`、`erase()` 或 `resize()`，但可以修改已有元素，也可以用 `fill()` 为所有元素赋值。

```cpp
std::array<int, 4> values{};
values[0] = 10;   // 正确：修改元素
values.resize(8); // 编译错误：不存在 resize
```

若整个容器应只读，应使用 `const std::array<T, N>`，而不是依赖“固定长度”产生错误直觉。

### 5. 复制与赋值处理全部元素

原生 C 数组不能直接复制或赋值，`std::array` 则具有普通值类型语义。复制会得到独立元素；函数按值接收 array 也会复制全部 `N` 个元素，因此只读参数通常使用 `const std::array<T, N> &`。

不同长度属于不同类型：`std::array<int, 3>` 不能赋值给 `std::array<int, 4>`。这让长度约束能参与 C++ 编译期类型检查。

In [ ]:
// 本步骤：通过代码演示“复制与赋值处理全部元素”并观察结果。
{
    // 复制 array；副本包含独立的三个元素。
    std::array<int, 3> original{10, 20, 30};
    std::array<int, 3> copy = original;

    // 只修改副本，再验证原对象的值没有随之改变。
    copy[0] = 99;

    assert((original == std::array<int, 3>{10, 20, 30}));
    assert((copy == std::array<int, 3>{99, 20, 30}));

    // 输出两个首元素，使独立值语义可以直接观察。
    std::cout << "original first = " << original.front() << '\n';
    std::cout << "copy first     = " << copy.front() << '\n';
}

### 6. `operator[]` 与 `at()` 的边界策略

`operator[]` 不检查索引，越界访问是 undefined behavior；`at()` 会检查边界并抛出 `std::out_of_range`。在纯 C++ 层可以根据错误策略选择，但异常不能直接穿过 C ABI。

`front()` 和 `back()` 只适用于非空 array。对于 `std::array<T, 0>`，`begin() == end()`，但调用 `front()`、`back()` 或解引用 `data()` 都不安全。

In [ ]:
// 本步骤：通过代码演示“operator[] 与 at() 的边界策略”并观察结果。
{
    // 创建非空 array，比较不检查边界的 [] 和检查边界的 at()。
    std::array<int, 3> values{10, 20, 30};

    assert(values[1] == 20);
    assert(values.at(2) == 30);

    // 主动访问越界位置，捕获 at() 报告的 C++ 异常。
    try
    {
        static_cast<void>(values.at(3));
    }
    catch (const std::out_of_range &error)
    {
        std::cout << "caught: " << error.what() << '\n';
    }

    // 构造合法的零长度 array，只验证空范围，不访问首尾元素。
    std::array<int, 0> empty{};
    assert(empty.begin() == empty.end());
    assert(empty.size() == 0);
}

### 7. 使用同一套 iterator 与标准算法

array 提供 `begin()/end()`，因此能与 vector 使用相同的标准算法。固定长度只限制结构，算法仍可排序、查找、转换或聚合元素。

标准算法通常接收半开区间 `[begin, end)`，所以同一算法不需要知道输入来自 array 还是 vector。后续的 `std::span` 会进一步把这种连续区间表达为非拥有视图。

In [ ]:
// 本步骤：通过代码演示“使用同一套 iterator 与标准算法”并观察结果。
{
    // 准备固定长度的输入和同长度的输出容器。
    std::array<int, 4> values{40, 10, 30, 20};
    std::array<int, 4> doubled{};

    // 通过 iterator 原地排序整个 array。
    std::sort(values.begin(), values.end());

    // 把每个已排序元素乘以二，并写入独立的输出 array。
    std::transform(
        values.cbegin(),
        values.cend(),
        doubled.begin(),
        [](int value)
        {
            return value * 2;
        });

    // 使用只读 iterator 聚合输入元素。
    const int sum = std::accumulate(
        values.cbegin(),
        values.cend(),
        0);

    // 验证三种算法结果并输出最终总和。
    assert((values == std::array<int, 4>{10, 20, 30, 40}));
    assert((doubled == std::array<int, 4>{20, 40, 60, 80}));
    assert(sum == 100);

    std::cout << "sum = " << sum << '\n';
}

### 8. 根据长度契约选择 array 或 vector

| 特性 | `std::array<T, N>` | `std::vector<T>` |
| --- | --- | --- |
| 元素数量 | 编译期固定，是类型的一部分 | 运行期可变 |
| 连续存储 | 是 | 是 |
| 单独动态分配 | 不需要 | 通常需要 |
| 增删元素 | 不支持 | 支持 |
| 重分配失效 | 不会发生 | capacity 不足时发生 |
| 典型场景 | 固定 header、坐标、颜色通道 | 文件内容、动态 payload、生成结果 |

选择标准不是“数据看起来很小”，而是 API 契约中的元素数量是否固定。大小可能变化时应使用 vector；固定协议字段、固定维度值或编译期表格适合 array。

### 9. C ABI 仍需显式传递或验证长度

`std::array` 是 C++ 类型，不能直接成为稳定 C ABI。即使 C 声明写成 `const uint8_t data[4]`，函数参数中的数组仍会退化为指针，编译器不会替调用方保证长度。

稳定边界通常继续使用 pointer + length，并在入口验证协议要求。下面模拟一个必须恰好接收四字节 header 的同步接口。

In [ ]:
// 本步骤：通过代码演示“C ABI 仍需显式传递或验证长度”并观察结果。
bool sdk_validate_header(
    const std::uint8_t *data,
    std::size_t size)
{
    // 在 C ABI 入口同时验证地址和运行期长度契约。
    if (data == nullptr || size != 4)
    {
        return false;
    }

    // 长度确认后才读取固定 header 的 magic bytes。
    return data[0] == 0x4b &&
           data[1] == 0x4e;
}

{
    // 由 array 持有四字节 header，确保 C++ 侧长度进入类型。
    const std::array<std::uint8_t, 4> header{
        0x4b,
        0x4e,
        0x01,
        0x00};

    // 在同步调用期间借出 pointer + length，不转移所有权。
    const bool valid =
        sdk_validate_header(header.data(), header.size());

    // 输出并断言边界校验结果。
    std::cout << "valid header = "
              << std::boolalpha
              << valid
              << '\n';

    assert(valid);
}

这里传出的 pointer 不拥有数据，只在调用期间借用 array：

```text
std::array<uint8_t, 4> owner
          │ data() + size()
          ▼
C ABI entry ── validate size == 4 ── read bytes
          │
          └─ return: borrow ends
```

如果 native 端需要在返回后或异步任务中保存数据，必须复制内容或建立明确的所有权协议，不能保存 `header.data()`。

Kotlin/Native 的 `ByteArray(4)` 长度仍是运行期状态，类型本身不会编码数字 4。因此 Kotlin wrapper 和 C ABI 入口都应验证长度；传入 native 的地址也只能在受约束的 pinned/调用作用域内借用。

### 10. 设计边界与常见误区

- 不要返回局部 array 的 `data()`，函数结束后 owner 已销毁。
- 不要把 `sizeof(std::array<T, N>)` 当作跨 ABI 序列化格式；对象表示、对齐和元素表示仍受类型与平台影响。
- `std::array<T, 0>` 是合法类型，但不能读取首尾元素。
- 固定长度不代表不可变，也不代表对象必然位于栈上。
- 对大型 array 按值传参会复制全部元素；只读借用使用常量引用，通用连续借用可在后续使用 `std::span`。

### 本实验结论

`std::array<T, N>` 是拥有固定数量连续元素的 RAII 值类型。长度 `N` 进入 C++ 类型系统，因此不同长度是不同类型；容器没有 capacity、扩容和结构性增删，也不会出现 vector 式重分配。

`data()`、reference 和 iterator 仍然不拥有元素，它们的有效期不能超过 array owner。固定长度只让存储地址更稳定，并没有取消生命周期规则。

跨 C ABI 或 Kotlin/Native 时，`std::array` 的编译期长度信息不会自动保留。边界仍需传递或验证长度，并明确借用只覆盖同步调用；长期保存和异步使用必须复制或建立所有权协议。